In [10]:
import requests
import json

url = "https://api.agmarknet.gov.in/v1/daily-price-arrival/filters"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Origin": "https://agmarknet.gov.in",
    "Referer": "https://agmarknet.gov.in/"
}

response = requests.get(url, headers=headers, timeout=30)
filters = response.json()

print("Response type:", type(filters))

if isinstance(filters, dict):
    print("\nAvailable keys:")
    print(list(filters.keys()))

    for key, value in filters.items():
        print(f"\n{key}:")
        print("Type:", type(value).__name__)

        if isinstance(value, list):
            print("Count:", len(value))
            print("Sample:", value[:3])
        elif isinstance(value, dict):
            print(json.dumps(value, indent=2)[:1500])
        else:
            print(str(value)[:500])

Response type: <class 'dict'>

Available keys:
['status', 'message', 'data']

status:
Type: bool
True

message:
Type: str
Filters fetched successfully

data:
Type: dict
{
  "cmdt_data": [
    {
      "cmdt_id": 380,
      "cmdt_name": "Absinthe",
      "cmdt_group_id": 12
    },
    {
      "cmdt_id": 571,
      "cmdt_name": "Adulsa",
      "cmdt_group_id": 16
    },
    {
      "cmdt_id": 454,
      "cmdt_name": "Ajwain Husk",
      "cmdt_group_id": 10
    },
    {
      "cmdt_id": 115,
      "cmdt_name": "Ajwan",
      "cmdt_group_id": 7
    },
    {
      "cmdt_id": 584,
      "cmdt_name": "Akarkara",
      "cmdt_group_id": 16
    },
    {
      "cmdt_id": 234,
      "cmdt_name": "Alasande Gram",
      "cmdt_group_id": 2
    },
    {
      "cmdt_id": 275,
      "cmdt_name": "Almond(Badam)",
      "cmdt_group_id": 8
    },
    {
      "cmdt_id": 455,
      "cmdt_name": "Aloe Vera",
      "cmdt_group_id": 6
    },
    {
      "cmdt_id": 591,
      "cmdt_name": "Aloe Vera Leaf",
      

In [11]:
import pandas as pd

data = filters["data"]

# Show all available filter categories
print("AVAILABLE FILTER CATEGORIES:")
print(list(data.keys()))

# Convert master data into DataFrames
for key, value in data.items():
    if isinstance(value, list):
        df = pd.DataFrame(value)

        print(f"\n{'='*60}")
        print("CATEGORY:", key)
        print("Records:", len(df))
        print("Columns:", df.columns.tolist())

        if any(x in key.lower() for x in ["cmdt", "state", "district"]):
            print(df.head(5).to_string(index=False))


# Find Wheat and Maize
commodity_df = pd.DataFrame(data["cmdt_data"])

print("\nWHEAT RECORDS:")
print(
    commodity_df[
        commodity_df["cmdt_name"].str.contains(
            "Wheat", case=False, na=False
        )
    ].to_string(index=False)
)

print("\nMAIZE RECORDS:")
print(
    commodity_df[
        commodity_df["cmdt_name"].str.contains(
            "Maize|Corn", case=False, na=False
        )
    ].to_string(index=False)
)

# Find Maharashtra
for key, value in data.items():
    if isinstance(value, list) and value:
        temp = pd.DataFrame(value)

        name_cols = [
            c for c in temp.columns
            if "state" in c.lower() and "name" in c.lower()
        ]

        for col in name_cols:
            result = temp[
                temp[col].astype(str).str.contains(
                    "Maharashtra", case=False, na=False
                )
            ]

            if not result.empty:
                print(f"\nMAHARASHTRA FOUND IN {key}:")
                print(result.to_string(index=False))

AVAILABLE FILTER CATEGORIES:
['cmdt_data', 'note_data', 'type_data', 'grade_data', 'range_data', 'state_data', 'market_data', 'variety_data', 'district_data', 'cmdt_group_data']

CATEGORY: cmdt_data
Records: 609
Columns: ['cmdt_id', 'cmdt_name', 'cmdt_group_id']
 cmdt_id   cmdt_name  cmdt_group_id
     380    Absinthe             12
     571      Adulsa             16
     454 Ajwain Husk             10
     115       Ajwan              7
     584    Akarkara             16

CATEGORY: note_data
Records: 2
Columns: ['id', 'note', 'type']

CATEGORY: type_data
Records: 3
Columns: ['id', 'type']

CATEGORY: grade_data
Records: 18
Columns: ['cmdt_id', 'grade_id', 'grade_name']

CATEGORY: range_data
Records: 1
Columns: ['both', 'price', 'arrival']

CATEGORY: state_data
Records: 37
Columns: ['state_id', 'state_name']
 state_id          state_name
   100000      All States/UTs
        1 Andaman and Nicobar
        2      Andhra Pradesh
        3   Arunachal Pradesh
        4               Assam

In [12]:
import pandas as pd
from pathlib import Path

data = filters["data"]

# 1. Maharashtra state ID
MH_STATE_ID = 20

# 2. District master
districts = pd.DataFrame(data["district_data"]).copy()

districts["state_id"] = pd.to_numeric(
    districts["state_id"], errors="coerce"
)

mh_districts = districts[
    districts["state_id"] == MH_STATE_ID
].copy()

mh_districts = mh_districts.rename(columns={
    "id": "district_id"
})

# 3. Market master
markets = pd.DataFrame(data["market_data"]).copy()

markets["state_id"] = pd.to_numeric(
    markets["state_id"], errors="coerce"
)

markets["district_id"] = pd.to_numeric(
    markets["district_id"], errors="coerce"
)

mh_markets = markets[
    markets["state_id"] == MH_STATE_ID
].copy()

# 4. Attach district names
mh_markets = mh_markets.merge(
    mh_districts[["district_id", "district_name"]],
    on="district_id",
    how="left"
)

# 5. Commodity master
commodities = pd.DataFrame(data["cmdt_data"])

target_commodities = commodities[
    commodities["cmdt_id"].isin([1, 4])
].copy()

# 6. Variety master
varieties = pd.DataFrame(data["variety_data"])

target_varieties = varieties[
    varieties["cmdt_id"].isin([1, 4])
].copy()

# 7. Grade master
grades = pd.DataFrame(data["grade_data"])

target_grades = grades[
    grades["cmdt_id"].isin([1, 4])
].copy()

# 8. Display summary
print("=" * 60)
print("MAHARASHTRA MASTER DATA SUMMARY")
print("=" * 60)

print("Districts:", len(mh_districts))
print("Markets:", len(mh_markets))
print("Wheat and Maize varieties:", len(target_varieties))
print("Wheat and Maize grades:", len(target_grades))

print("\nDISTRICTS:")
print(
    mh_districts[
        ["district_id", "district_name"]
    ].sort_values("district_name").to_string(index=False)
)

print("\nMARKET SAMPLE:")
print(mh_markets.head(10).to_string(index=False))

print("\nCOMMODITIES:")
print(target_commodities.to_string(index=False))

# 9. Save reference files
output_dir = Path("agmarknet_master_data")
output_dir.mkdir(exist_ok=True)

mh_districts.to_csv(
    output_dir / "maharashtra_districts.csv", index=False
)

mh_markets.to_csv(
    output_dir / "maharashtra_markets.csv", index=False
)

target_commodities.to_csv(
    output_dir / "wheat_maize_commodities.csv", index=False
)

target_varieties.to_csv(
    output_dir / "wheat_maize_varieties.csv", index=False
)

target_grades.to_csv(
    output_dir / "wheat_maize_grades.csv", index=False
)

print("\nMaster files saved:", output_dir.resolve())

MAHARASHTRA MASTER DATA SUMMARY
Districts: 38
Markets: 429
Wheat and Maize varieties: 0
Wheat and Maize grades: 0

DISTRICTS:
 district_id             district_name
         338               Ahilyanagar
         339                     Akola
         340                 Amarawati
         341                 Bandra(E)
         342                      Beed
         343                  Bhandara
         344                  Buldhana
         345                Chandrapur
         346 Chattrapati Sambhajinagar
         347                 Dharashiv
         348                     Dhule
         349                Gadchiroli
         350                   Gondiya
         351                   Hingoli
         353                   Jalgaon
         352                     Jalna
         354                  Kolhapur
         355                     Latur
         356                    Mumbai
         357                     Murum
         358                    Nagpur
         359    

In [13]:
import requests

url = "https://api.ceda.ashoka.edu.in/openapi.json"

r = requests.get(url, timeout=30)

print("Status:", r.status_code)

if r.status_code == 200:
    spec = r.json()

    for path, methods in spec.get("paths", {}).items():
        if "agmarknet" in path.lower():
            print("\n", path)

            for method, details in methods.items():
                print("Method:", method.upper())
                print("Summary:", details.get("summary"))

                for param in details.get("parameters", []):
                    print(
                        "Parameter:",
                        param.get("name"),
                        param.get("required"),
                        param.get("schema")
                    )

                if "requestBody" in details:
                    print(
                        "Request body:",
                        details["requestBody"]
                    )
else:
    print(r.text[:2000])

Status: 404
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot GET /openapi.json</pre>
</body>
</html>



In [15]:
import requests
import json

BASE_URL = "https://agmarknet.ceda.ashoka.edu.in/api"

endpoints = [
    "/commodities",
    "/states"
]

for endpoint in endpoints:

    print("\n" + "=" * 60)
    print("TESTING:", endpoint)
    print("=" * 60)

    try:
        response = requests.get(
            BASE_URL + endpoint,
            timeout=30
        )

        print("HTTP Status:", response.status_code)

        if response.status_code == 200:
            result = response.json()

            print(json.dumps(result, indent=2)[:3000])

        else:
            print(response.text[:1000])

    except Exception as e:
        print("ERROR:", repr(e))


TESTING: /commodities
ERROR: ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='agmarknet.ceda.ashoka.edu.in', port=443): Max retries exceeded with url: /api/commodities (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000023420DB87D0>, 'Connection to agmarknet.ceda.ashoka.edu.in timed out. (connect timeout=30)'))"))

TESTING: /states
ERROR: ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='agmarknet.ceda.ashoka.edu.in', port=443): Max retries exceeded with url: /api/states (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000023420DBA0D0>, 'Connection to agmarknet.ceda.ashoka.edu.in timed out. (connect timeout=30)'))"))


In [16]:

import requests
import pandas as pd

API_KEY = "579b464db66ec23bdd000001f5961e2a74914c76781b5b89945bb5af"

RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"

url = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

params = {
    "api-key": API_KEY,
    "format": "json",
    "limit": 10,
    "offset": 0,
    "filters[state]": "Maharashtra",
    "filters[commodity]": "Wheat"
}

try:
    response = requests.get(url, params=params, timeout=30)

    print("HTTP Status:", response.status_code)

    response.raise_for_status()

    result = response.json()

    print("\nTotal matching records:", result.get("total"))
    print("\nAvailable fields:")
    print([field["id"] for field in result.get("field", [])])

    df_test = pd.DataFrame(result.get("records", []))

    print("\nSample records:")
    display(df_test)

except Exception as e:
    print("ERROR:", repr(e))


ERROR: ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=30)"))


In [18]:

import requests
import time

urls = [
    "https://www.data.gov.in/",
    "https://api.data.gov.in/",
    "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"
]

for url in urls:
    print("\n" + "=" * 60)
    print("TESTING:", url)

    try:
        start = time.time()

        response = requests.get(
            url,
            timeout=(10, 60),
            headers={"User-Agent": "Mozilla/5.0"}
        )

        elapsed = round(time.time() - start, 2)

        print("HTTP Status:", response.status_code)
        print("Time taken:", elapsed, "seconds")
        print("Response:", response.text[:300])

    except requests.exceptions.ConnectTimeout:
        print("CONNECTION TIMEOUT")

    except requests.exceptions.ReadTimeout:
        print("SERVER RESPONSE TIMEOUT")

    except requests.exceptions.RequestException as e:
        print("CONNECTION ERROR:", repr(e))



TESTING: https://www.data.gov.in/
HTTP Status: 200
Time taken: 0.7 seconds
Response: <!doctype html>
<html data-n-head-ssr lang="en" data-n-head="%7B%22lang%22:%7B%22ssr%22:%22en%22%7D%7D">
  <head >
    <meta data-n-head="ssr" charset="utf-8"><meta data-n-head="ssr" name="http-equiv" content="text/html"><meta data-n-head="ssr" name="viewport" content="width=device-width, initial-sc

TESTING: https://api.data.gov.in/
HTTP Status: 404
Time taken: 0.38 seconds
Response: 404 page not found


TESTING: https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070
HTTP Status: 400
Time taken: 0.37 seconds
Response: {
    "error": "Authorization field missing"
}


In [21]:

import os
import requests
import pandas as pd

API_KEY = os.getenv("579b464db66ec23bdd000001f5961e2a74914c76781b5b89945bb5af")

RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"

url = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

params = {
    "api-key": API_KEY,
    "format": "json",
    "limit": 10,
    "offset": 0,
    "filters[state]": "Maharashtra",
    "filters[commodity]": "Wheat"
}

try:
    response = requests.get(
        url,
        params=params,
        timeout=(15, 90)
    )

    print("Status:", response.status_code)
    print("Response:", response.text[:1000])

    response.raise_for_status()

    result = response.json()
    df_test = pd.DataFrame(result.get("records", []))

    print("Total records:", result.get("total"))
    display(df_test)

except Exception as e:
    print("ERROR:", repr(e))


Status: 502
Response: 
ERROR: HTTPError('502 Server Error: Bad Gateway for url: https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070?format=json&limit=10&offset=0&filters%5Bstate%5D=Maharashtra&filters%5Bcommodity%5D=Wheat')


In [22]:

import os
import requests
import pandas as pd

API_KEY = os.getenv("579b464db66ec23bdd000001f5961e2a74914c76781b5b89945bb5af")

if not API_KEY:
    raise ValueError("API key not found.")

RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"

url = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Accept": "application/json"
}

params = {
    "format": "json",
    "limit": 10,
    "offset": 0,
    "filters[state]": "Maharashtra",
    "filters[commodity]": "Wheat"
}

try:
    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=(15, 60)
    )

    print("HTTP Status:", response.status_code)
    print("Response:", response.text[:2000])

    response.raise_for_status()

    result = response.json()

    print("\nTotal records:", result.get("total"))
    print("\nFields:", [x["id"] for x in result.get("field", [])])

    df_test = pd.DataFrame(result.get("records", []))
    display(df_test)

except Exception as e:
    print("ERROR:", repr(e))


ValueError: API key not found.

In [24]:
import requests
import os

API_KEY = os.getenv("579b464db66ec23bdd000001f5961e2a74914c76781b5b89945bb5af")

RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"

url = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

params = {
    "api-key": API_KEY,
    "format": "json",
    "limit": 10,
    "offset": 0,
    "filters[state]": "Maharashtra",
    "filters[commodity]": "Wheat"
}

response = requests.get(url, params=params, timeout=60)

print("Status:", response.status_code)
print("Response:", response.text[:1000])

ReadTimeout: HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=60)